In [2]:
import pandas as pd
import numpy as np
import time
import platform
import os

print("Inizializzazione Motore di Scansione - SETUP 2: IMBALANCE / SWEEP...")
start_time = time.time()

# --- 1. CARICAMENTO DATI CROSS-PLATFORM (Mac/Windows) ---
# Il codice riconosce automaticamente il sistema operativo
sistema_operativo = platform.system()

if sistema_operativo == "Windows":
    percorso_parquet = r"C:\Users\Leona\source\repos\QuantEdge_Project\data\processed\es_master_4months.parquet"
    print("Rilevato sistema: WINDOWS")
else:
    percorso_parquet = "/Users/leonardosposato/Documents/git/QuantEdge_Project/data/processed/es_master_4months.parquet"
    print("Rilevato sistema: MAC/LINUX")

# Verifica aggiuntiva per evitare che il programma esploda se il file non esiste
if not os.path.exists(percorso_parquet):
    raise FileNotFoundError(f"🚨 ERRORE CRITICO: Il file non esiste nel percorso: {percorso_parquet}")

print(f"Caricamento dataset in corso...")
df = pd.read_parquet(percorso_parquet).dropna(subset=['bid', 'ask'])

# Rendiamo la colonna temporale UTC-Aware a prova di proiettile
df['Datetime_UTC'] = pd.to_datetime(df['Datetime_UTC'], utc=True)
df = df.set_index('Datetime_UTC').sort_index()

print(f"Dati caricati. Righe operative: {len(df):,}")

# --- 2. PARAMETRI SETUP 2 (Definiti da Claude) ---
FINESTRA_SEC = '2s'           # Finestra ultra-rapida di 2 secondi
SOGLIA_VOLUME = 800           # Esplosione di volume massiccia
SOGLIA_DELTA_RATIO = 0.70     # Almeno il 70% dei volumi deve colpire una sola direzione
MIN_PRICE_MOVE = 0.50         # Lo strappo deve muovere il prezzo di almeno 2 tick interi

print(f"Calcolo aggregazioni Rolling a {FINESTRA_SEC}...")

# --- 3. COSTRUZIONE DEI SEGNALI (VETTORIALE) ---
df['Rolling_Vol_2s'] = df['Volume'].rolling(window=FINESTRA_SEC).sum()
df['Rolling_Delta_2s'] = df['Delta'].rolling(window=FINESTRA_SEC).sum()
df['Rolling_Max_2s'] = df['Price'].rolling(window=FINESTRA_SEC).max()
df['Rolling_Min_2s'] = df['Price'].rolling(window=FINESTRA_SEC).min()
df['Price_Range_2s'] = df['Rolling_Max_2s'] - df['Rolling_Min_2s']

# Calcolo del Delta Ratio (protezione da divisione per zero)
df['Delta_Ratio'] = np.where(
    df['Rolling_Vol_2s'] > 0, 
    abs(df['Rolling_Delta_2s']) / df['Rolling_Vol_2s'], 
    0
)

print("Ricerca degli squilibri istituzionali (Sweep)...")

# TRIGGER LONG (Sweep istituzionale in acquisto)
cond_long = (
    (df['Rolling_Vol_2s'] >= SOGLIA_VOLUME) &
    (df['Rolling_Delta_2s'] > 0) &
    (df['Delta_Ratio'] >= SOGLIA_DELTA_RATIO) &
    (df['Price_Range_2s'] >= MIN_PRICE_MOVE)
)

# TRIGGER SHORT (Sweep istituzionale in vendita)
cond_short = (
    (df['Rolling_Vol_2s'] >= SOGLIA_VOLUME) &
    (df['Rolling_Delta_2s'] < 0) &
    (df['Delta_Ratio'] >= SOGLIA_DELTA_RATIO) &
    (df['Price_Range_2s'] >= MIN_PRICE_MOVE)
)

# Registriamo il segnale
df['Signal_Imbalance'] = np.where(cond_long, 1, np.where(cond_short, -1, 0))

# Isoliamo solo le anomalie trovate
imbalance_signals = df[df['Signal_Imbalance'] != 0].copy()

# Pulizia memoria
df = df.drop(columns=['Rolling_Vol_2s', 'Rolling_Delta_2s', 'Rolling_Max_2s', 'Rolling_Min_2s', 'Price_Range_2s', 'Delta_Ratio', 'Signal_Imbalance'])

print(f"\n--- SCANSIONE COMPLETATA IN {(time.time() - start_time):.2f} SECONDI ---")
print(f"Squilibri Istituzionali Trovati (Grezzi): {len(imbalance_signals):,}")

# Mostriamo un'anteprima dei segnali grezzi trovati
if len(imbalance_signals) > 0:
    display(imbalance_signals[['Price', 'Rolling_Vol_2s', 'Rolling_Delta_2s', 'Delta_Ratio', 'Price_Range_2s', 'Signal_Imbalance']].head(10))

Inizializzazione Motore di Scansione - SETUP 2: IMBALANCE / SWEEP...
Rilevato sistema: WINDOWS
Caricamento dataset in corso...
Dati caricati. Righe operative: 71,651,488
Calcolo aggregazioni Rolling a 2s...
Ricerca degli squilibri istituzionali (Sweep)...

--- SCANSIONE COMPLETATA IN 43.79 SECONDI ---
Squilibri Istituzionali Trovati (Grezzi): 35,906


,Price,Rolling_Vol_2s,Rolling_Delta_2s,Delta_Ratio,Price_Range_2s,Signal_Imbalance
Datetime_UTC,,,,,,
2026-02-11 19:19:36+00:00,6973.75,800.0,640.0,0.800000,0.75,1
2026-02-11 19:19:36+00:00,6973.75,801.0,641.0,0.800250,0.75,1
2026-02-11 19:19:36+00:00,6973.75,802.0,642.0,0.800499,0.75,1
2026-02-11 19:19:36+00:00,6973.75,803.0,643.0,0.800747,0.75,1
2026-02-11 19:19:36+00:00,6973.75,804.0,644.0,0.800995,0.75,1
2026-02-11 19:19:36+00:00,6973.75,805.0,645.0,0.801242,0.75,1
2026-02-11 19:19:36+00:00,6973.75,806.0,646.0,0.801489,0.75,1
2026-02-11 19:19:36+00:00,6973.75,807.0,645.0,0.799257,0.75,1
2026-02-11 19:19:36+00:00,6973.75,808.0,644.0,0.797030,0.75,1


In [3]:
import pandas as pd
import numpy as np
import time

if 'imbalance_signals' not in globals() or 'df' not in globals():
    raise NameError("Errore: Esegui questa cella nel notebook originale che contiene i dati!")

print("Avvio Motore di Esecuzione SMART (Setup 2: Imbalance + Pullback)...")
start_sim = time.time()

# --- 1. FILTRO CLUSTER (COOLDOWN) ---
COOLDOWN_SEC = 15  # Pausa di 15 secondi tra un'esplosione e l'altra

signals_df = imbalance_signals.reset_index()
valid_sweeps = []
last_sweep_time = None

for idx, row in signals_df.iterrows():
    if last_sweep_time is None or (row['Datetime_UTC'] - last_sweep_time).total_seconds() >= COOLDOWN_SEC:
        valid_sweeps.append(row)
        last_sweep_time = row['Datetime_UTC']

df_sweeps = pd.DataFrame(valid_sweeps)
print(f"Segnali Grezzi: {len(imbalance_signals):,} -> Esplosioni Uniche: {len(df_sweeps):,}")

# --- 2. RISK ENGINE CON PULLBACK (Regole di Claude) ---
print("Simulazione Entrata su Ritracciamento e Gestione Trade...")

WAIT_PULLBACK_SEC = 3.0    # Secondi massimi per aspettare il ritracciamento
PULLBACK_PT = 0.25         # Entriamo solo se il mercato ci fa lo "sconto" di 1 tick
TP_PUNTI = 1.00            # Target Profit
SL_PUNTI = 0.75            # Stop Loss (stretto, perché se il momentum muore usciamo)
TIME_STOP_SEC = 20         # Il trade deve chiudersi in 20 secondi

esiti = []
pnls = []
trade_eseguiti = 0

for _, sweep in df_sweeps.iterrows():
    t_burst = sweep['Datetime_UTC']
    direzione = sweep['Signal_Imbalance']
    prezzo_burst = sweep['Price']
    
    # -- FASE A: CERCHIAMO IL PULLBACK NEI 3 SECONDI SUCCESSIVI --
    t_fine_pullback = t_burst + pd.Timedelta(seconds=WAIT_PULLBACK_SEC)
    finestra_pullback = df.loc[t_burst : t_fine_pullback]
    
    if finestra_pullback.empty:
        continue
        
    t_entrata = None
    prezzo_entrata = None
    
    if direzione == 1: # LONG
        # Vogliamo comprare a un prezzo PIÙ BASSO del picco (-0.25 pt)
        prezzo_target_pb = prezzo_burst - PULLBACK_PT
        # Cerchiamo il primo millisecondo in cui l'Ask ci fa questo prezzo
        hit_pb = finestra_pullback[finestra_pullback['ask'] <= prezzo_target_pb]
        if not hit_pb.empty:
            t_entrata = hit_pb.index[0]
            prezzo_entrata = prezzo_target_pb
            
    elif direzione == -1: # SHORT
        # Vogliamo vendere a un prezzo PIÙ ALTO del picco (+0.25 pt)
        prezzo_target_pb = prezzo_burst + PULLBACK_PT
        # Cerchiamo il primo millisecondo in cui il Bid ci fa questo prezzo
        hit_pb = finestra_pullback[finestra_pullback['bid'] >= prezzo_target_pb]
        if not hit_pb.empty:
            t_entrata = hit_pb.index[0]
            prezzo_entrata = prezzo_target_pb

    # Se il ritracciamento non è avvenuto entro 3 secondi, skippiamo il trade (Missed Opportunity)
    if not t_entrata:
        esiti.append('MISSED_PULLBACK')
        pnls.append(0)
        continue
        
    trade_eseguiti += 1
        
    # -- FASE B: GESTIONE DEL TRADE (SL / TP / TIME STOP) --
    t_scadenza = t_entrata + pd.Timedelta(seconds=TIME_STOP_SEC)
    finestra_trade = df.loc[t_entrata : t_scadenza]
    
    if finestra_trade.empty:
        esiti.append('TIME_STOP'); pnls.append(0)
        continue

    if direzione == 1: # GESTIONE LONG
        target = prezzo_entrata + TP_PUNTI
        stop = prezzo_entrata - SL_PUNTI
        
        hit_tp = (finestra_trade['bid'] >= target).idxmax() if (finestra_trade['bid'] >= target).any() else None
        hit_sl = (finestra_trade['bid'] <= stop).idxmax() if (finestra_trade['bid'] <= stop).any() else None
        
        if hit_tp and hit_sl:
            if hit_tp < hit_sl: esiti.append('WIN'); pnls.append(TP_PUNTI)
            else: esiti.append('LOSS'); pnls.append(-SL_PUNTI)
        elif hit_tp: esiti.append('WIN'); pnls.append(TP_PUNTI)
        elif hit_sl: esiti.append('LOSS'); pnls.append(-SL_PUNTI)
        else:
            esiti.append('TIME_STOP')
            pnls.append(finestra_trade['bid'].iloc[-1] - prezzo_entrata)

    elif direzione == -1: # GESTIONE SHORT
        target = prezzo_entrata - TP_PUNTI
        stop = prezzo_entrata + SL_PUNTI
        
        hit_tp = (finestra_trade['ask'] <= target).idxmax() if (finestra_trade['ask'] <= target).any() else None
        hit_sl = (finestra_trade['ask'] >= stop).idxmax() if (finestra_trade['ask'] >= stop).any() else None
        
        if hit_tp and hit_sl:
            if hit_tp < hit_sl: esiti.append('WIN'); pnls.append(TP_PUNTI)
            else: esiti.append('LOSS'); pnls.append(-SL_PUNTI)
        elif hit_tp: esiti.append('WIN'); pnls.append(TP_PUNTI)
        elif hit_sl: esiti.append('LOSS'); pnls.append(-SL_PUNTI)
        else:
            esiti.append('TIME_STOP')
            pnls.append(prezzo_entrata - finestra_trade['ask'].iloc[-1])

df_sweeps['Esito'] = esiti
df_sweeps['PnL'] = pnls

# --- 3. REPORT FINALE ---
wins = len(df_sweeps[df_sweeps['Esito'] == 'WIN'])
losses = len(df_sweeps[df_sweeps['Esito'] == 'LOSS'])
missed = len(df_sweeps[df_sweeps['Esito'] == 'MISSED_PULLBACK'])
time_stops = len(df_sweeps[df_sweeps['Esito'] == 'TIME_STOP'])
net_pnl = sum(pnls)

win_rate = (wins / trade_eseguiti) * 100 if trade_eseguiti > 0 else 0

print(f"\n--- REPORT SETUP 2 (IMBALANCE + PULLBACK ENTRY) ---")
print(f"Esplosioni Uniche Rilevate: {len(df_sweeps)}")
print(f"Trade ESEGUITI (Ritracciamento Hit): {trade_eseguiti}")
print(f"Trade MANCATI (No Ritracciamento): {missed}")
print(f"---------------------------------------------------")
print(f"Vittorie (TP Hit): {wins}")
print(f"Sconfitte (SL Hit): {losses}")
print(f"Uscite per Tempo (Time Stop): {time_stops}")
print(f"Win Rate Reale (Sui trade eseguiti): {win_rate:.2f}%")
print(f"Net PnL (in punti indice): {net_pnl:.2f} Punti")
print(f"Profitto Lordo Stimato (1 Lotto ES): ${net_pnl * 50:,.2f}")
print(f"Tempo di calcolo: {(time.time() - start_sim):.2f} sec")

Avvio Motore di Esecuzione SMART (Setup 2: Imbalance + Pullback)...
Segnali Grezzi: 35,906 -> Esplosioni Uniche: 263
Simulazione Entrata su Ritracciamento e Gestione Trade...

--- REPORT SETUP 2 (IMBALANCE + PULLBACK ENTRY) ---
Esplosioni Uniche Rilevate: 263
Trade ESEGUITI (Ritracciamento Hit): 152
Trade MANCATI (No Ritracciamento): 111
---------------------------------------------------
Vittorie (TP Hit): 0
Sconfitte (SL Hit): 152
Uscite per Tempo (Time Stop): 0
Win Rate Reale (Sui trade eseguiti): 0.00%
Net PnL (in punti indice): -114.00 Punti
Profitto Lordo Stimato (1 Lotto ES): $-5,700.00
Tempo di calcolo: 2.74 sec


In [4]:
import pandas as pd

if 'df_sweeps' not in globals() or 'df' not in globals():
    raise NameError("Errore: Esegui questa cella nel notebook originale che contiene i dati!")

print("--- DEBUG VISIVO: AUTopsia del primo trade fallito ---")

# Prendiamo solo i trade in cui siamo effettivamente entrati a mercato e abbiamo perso
trade_eseguiti = df_sweeps[df_sweeps['Esito'] == 'LOSS'].copy()

if trade_eseguiti.empty:
    print("Nessun trade fallito da analizzare.")
else:
    # Prendiamo il primissimo trade della lista
    trade_test = trade_eseguiti.iloc[0]
    
    t_burst = trade_test['Datetime_UTC']
    direzione = trade_test['Signal_Imbalance']
    prezzo_burst = trade_test['Price']
    
    print(f"\n[1] L'ESPLOSIONE")
    print(f"Orario Sweep: {t_burst}")
    print(f"Direzione: {'LONG' if direzione == 1 else 'SHORT'} (Segnale: {direzione})")
    print(f"Prezzo Picco Sweep: {prezzo_burst}")
    
    # 1. Trovare il momento dell'entrata (Pullback)
    WAIT_PULLBACK_SEC = 3.0
    PULLBACK_PT = 0.25
    t_fine_pullback = t_burst + pd.Timedelta(seconds=WAIT_PULLBACK_SEC)
    finestra_pullback = df.loc[t_burst : t_fine_pullback]
    
    t_entrata = None
    prezzo_entrata = None
    
    if direzione == 1:
        prezzo_target_pb = prezzo_burst - PULLBACK_PT
        hit_pb = finestra_pullback[finestra_pullback['ask'] <= prezzo_target_pb]
        if not hit_pb.empty:
            t_entrata = hit_pb.index[0]
            prezzo_entrata = prezzo_target_pb
            prezzo_reale_ask = hit_pb['ask'].iloc[0]
            prezzo_reale_bid = hit_pb['bid'].iloc[0]
            
    elif direzione == -1:
        prezzo_target_pb = prezzo_burst + PULLBACK_PT
        hit_pb = finestra_pullback[finestra_pullback['bid'] >= prezzo_target_pb]
        if not hit_pb.empty:
            t_entrata = hit_pb.index[0]
            prezzo_entrata = prezzo_target_pb
            prezzo_reale_ask = hit_pb['ask'].iloc[0]
            prezzo_reale_bid = hit_pb['bid'].iloc[0]
            
    print(f"\n[2] L'INGRESSO A MERCATO (PULLBACK)")
    print(f"Orario Ingresso: {t_entrata}")
    print(f"Prezzo Target Pullback: {prezzo_entrata}")
    print(f"Bid Reale al momento dell'ingresso: {prezzo_reale_bid}")
    print(f"Ask Reale al momento dell'ingresso: {prezzo_reale_ask}")
    print(f"Spread Reale (Ask - Bid): {(prezzo_reale_ask - prezzo_reale_bid):.2f} Punti")
    
    # 2. Trovare il momento dello Stop Loss
    TP_PUNTI = 1.00
    SL_PUNTI = 0.75
    TIME_STOP_SEC = 20
    
    t_scadenza = t_entrata + pd.Timedelta(seconds=TIME_STOP_SEC)
    finestra_trade = df.loc[t_entrata : t_scadenza]
    
    print(f"\n[3] L'ESECUZIONE (TP/SL)")
    if direzione == 1:
        target = prezzo_entrata + TP_PUNTI
        stop = prezzo_entrata - SL_PUNTI
        print(f"Target Profit impostato a: {target} (da colpire sul BID)")
        print(f"Stop Loss impostato a: {stop} (da colpire sul BID)")
        
        # Mostriamo i primi 10 tick di mercato subito dopo l'ingresso
        print("\nPrimi 10 millisecondi successivi all'ingresso (Come si è mosso il Bid?):")
        display(finestra_trade[['bid', 'ask']].head(10))
        
    elif direzione == -1:
        target = prezzo_entrata - TP_PUNTI
        stop = prezzo_entrata + SL_PUNTI
        print(f"Target Profit impostato a: {target} (da colpire sull'ASK)")
        print(f"Stop Loss impostato a: {stop} (da colpire sull'ASK)")
        
        # Mostriamo i primi 10 tick di mercato subito dopo l'ingresso
        print("\nPrimi 10 millisecondi successivi all'ingresso (Come si è mosso l'Ask?):")
        display(finestra_trade[['bid', 'ask']].head(10))

--- DEBUG VISIVO: AUTopsia del primo trade fallito ---

[1] L'ESPLOSIONE
Orario Sweep: 2026-02-11 19:19:36+00:00
Direzione: LONG (Segnale: 1)
Prezzo Picco Sweep: 6973.75

[2] L'INGRESSO A MERCATO (PULLBACK)
Orario Ingresso: 2026-02-11 19:19:36+00:00
Prezzo Target Pullback: 6973.5
Bid Reale al momento dell'ingresso: 6948.9
Ask Reale al momento dell'ingresso: 6949.15
Spread Reale (Ask - Bid): 0.25 Punti

[3] L'ESECUZIONE (TP/SL)
Target Profit impostato a: 6974.5 (da colpire sul BID)
Stop Loss impostato a: 6972.75 (da colpire sul BID)

Primi 10 millisecondi successivi all'ingresso (Come si è mosso il Bid?):


,bid,ask
Datetime_UTC,,
2026-02-11 19:19:36+00:00,6948.9,6949.15
2026-02-11 19:19:36+00:00,6948.9,6949.15
2026-02-11 19:19:36+00:00,6948.9,6949.15
2026-02-11 19:19:36+00:00,6948.9,6949.15
2026-02-11 19:19:36+00:00,6948.9,6949.15
2026-02-11 19:19:36+00:00,6948.9,6949.15
2026-02-11 19:19:36+00:00,6948.9,6949.15
2026-02-11 19:19:36+00:00,6948.9,6949.15
2026-02-11 19:19:36+00:00,6948.9,6949.15


In [5]:
import pandas as pd
import numpy as np
import time

if 'imbalance_signals' not in globals() or 'df' not in globals():
    raise NameError("Errore: Esegui questa cella nel notebook originale che contiene i dati!")

print("Avvio Motore di Esecuzione SMART (V2 - Ancoraggio Sicuro al Bid/Ask)...")
start_sim = time.time()

# --- 1. FILTRO CLUSTER (COOLDOWN) ---
COOLDOWN_SEC = 15  

signals_df = imbalance_signals.reset_index()
valid_sweeps = []
last_sweep_time = None

for idx, row in signals_df.iterrows():
    if last_sweep_time is None or (row['Datetime_UTC'] - last_sweep_time).total_seconds() >= COOLDOWN_SEC:
        valid_sweeps.append(row)
        last_sweep_time = row['Datetime_UTC']

df_sweeps = pd.DataFrame(valid_sweeps)
print(f"Segnali Grezzi: {len(imbalance_signals):,} -> Esplosioni Uniche: {len(df_sweeps):,}")

# --- 2. RISK ENGINE CON PULLBACK ANCORATO ---
WAIT_PULLBACK_SEC = 3.0    
PULLBACK_PT = 0.25         
TP_PUNTI = 1.00            
SL_PUNTI = 0.75            
TIME_STOP_SEC = 20         

esiti = []
pnls = []
trade_eseguiti = 0

for _, sweep in df_sweeps.iterrows():
    t_burst = sweep['Datetime_UTC']
    direzione = sweep['Signal_Imbalance']
    
    t_fine_pullback = t_burst + pd.Timedelta(seconds=WAIT_PULLBACK_SEC)
    finestra_pullback = df.loc[t_burst : t_fine_pullback]
    
    if finestra_pullback.empty:
        continue
        
    t_entrata = None
    prezzo_entrata = None
    
    if direzione == 1: # LONG
        # FIX: Calcoliamo il pullback a partire dall'Ask reale, ignorando il 'Price' fallato
        prezzo_picco_ask = sweep['ask'] 
        prezzo_target_pb = prezzo_picco_ask - PULLBACK_PT
        
        hit_pb = finestra_pullback[finestra_pullback['ask'] <= prezzo_target_pb]
        if not hit_pb.empty:
            t_entrata = hit_pb.index[0]
            # Registriamo l'ingresso all'Ask REALE
            prezzo_entrata = hit_pb['ask'].iloc[0] 
            
    elif direzione == -1: # SHORT
        # FIX: Calcoliamo il pullback a partire dal Bid reale
        prezzo_picco_bid = sweep['bid'] 
        prezzo_target_pb = prezzo_picco_bid + PULLBACK_PT
        
        hit_pb = finestra_pullback[finestra_pullback['bid'] >= prezzo_target_pb]
        if not hit_pb.empty:
            t_entrata = hit_pb.index[0]
            # Registriamo l'ingresso al Bid REALE
            prezzo_entrata = hit_pb['bid'].iloc[0] 

    if not t_entrata:
        esiti.append('MISSED_PULLBACK')
        pnls.append(0)
        continue
        
    trade_eseguiti += 1
        
    # --- FASE B: GESTIONE DEL TRADE ---
    t_scadenza = t_entrata + pd.Timedelta(seconds=TIME_STOP_SEC)
    finestra_trade = df.loc[t_entrata : t_scadenza]
    
    if finestra_trade.empty:
        esiti.append('TIME_STOP'); pnls.append(0)
        continue

    if direzione == 1: # GESTIONE LONG
        target = prezzo_entrata + TP_PUNTI
        stop = prezzo_entrata - SL_PUNTI
        
        hit_tp = (finestra_trade['bid'] >= target).idxmax() if (finestra_trade['bid'] >= target).any() else None
        hit_sl = (finestra_trade['bid'] <= stop).idxmax() if (finestra_trade['bid'] <= stop).any() else None
        
        if hit_tp and hit_sl:
            if hit_tp < hit_sl: esiti.append('WIN'); pnls.append(target - prezzo_entrata)
            else: esiti.append('LOSS'); pnls.append(stop - prezzo_entrata)
        elif hit_tp: esiti.append('WIN'); pnls.append(target - prezzo_entrata)
        elif hit_sl: esiti.append('LOSS'); pnls.append(stop - prezzo_entrata)
        else:
            esiti.append('TIME_STOP')
            pnls.append(finestra_trade['bid'].iloc[-1] - prezzo_entrata)

    elif direzione == -1: # GESTIONE SHORT
        target = prezzo_entrata - TP_PUNTI
        stop = prezzo_entrata + SL_PUNTI
        
        hit_tp = (finestra_trade['ask'] <= target).idxmax() if (finestra_trade['ask'] <= target).any() else None
        hit_sl = (finestra_trade['ask'] >= stop).idxmax() if (finestra_trade['ask'] >= stop).any() else None
        
        if hit_tp and hit_sl:
            if hit_tp < hit_sl: esiti.append('WIN'); pnls.append(prezzo_entrata - target)
            else: esiti.append('LOSS'); pnls.append(prezzo_entrata - stop)
        elif hit_tp: esiti.append('WIN'); pnls.append(prezzo_entrata - target)
        elif hit_sl: esiti.append('LOSS'); pnls.append(prezzo_entrata - stop)
        else:
            esiti.append('TIME_STOP')
            pnls.append(prezzo_entrata - finestra_trade['ask'].iloc[-1])

df_sweeps['Esito'] = esiti
df_sweeps['PnL'] = pnls

# --- 3. REPORT FINALE ---
wins = len(df_sweeps[df_sweeps['Esito'] == 'WIN'])
losses = len(df_sweeps[df_sweeps['Esito'] == 'LOSS'])
missed = len(df_sweeps[df_sweeps['Esito'] == 'MISSED_PULLBACK'])
time_stops = len(df_sweeps[df_sweeps['Esito'] == 'TIME_STOP'])
net_pnl = sum(pnls)

win_rate = (wins / trade_eseguiti) * 100 if trade_eseguiti > 0 else 0

print(f"\n--- REPORT SETUP 2 (BUG DATASET FIXATO) ---")
print(f"Esplosioni Uniche Rilevate: {len(df_sweeps)}")
print(f"Trade ESEGUITI (Ritracciamento Hit): {trade_eseguiti}")
print(f"Trade MANCATI (No Ritracciamento): {missed}")
print(f"---------------------------------------------------")
print(f"Vittorie (TP Hit): {wins}")
print(f"Sconfitte (SL Hit): {losses}")
print(f"Uscite per Tempo (Time Stop): {time_stops}")
print(f"Win Rate Reale (Sui trade eseguiti): {win_rate:.2f}%")
print(f"Net PnL (in punti indice): {net_pnl:.2f} Punti")
print(f"Tempo di calcolo: {(time.time() - start_sim):.2f} sec")

Avvio Motore di Esecuzione SMART (V2 - Ancoraggio Sicuro al Bid/Ask)...
Segnali Grezzi: 35,906 -> Esplosioni Uniche: 263

--- REPORT SETUP 2 (BUG DATASET FIXATO) ---
Esplosioni Uniche Rilevate: 263
Trade ESEGUITI (Ritracciamento Hit): 116
Trade MANCATI (No Ritracciamento): 147
---------------------------------------------------
Vittorie (TP Hit): 24
Sconfitte (SL Hit): 70
Uscite per Tempo (Time Stop): 22
Win Rate Reale (Sui trade eseguiti): 20.69%
Net PnL (in punti indice): -24.66 Punti
Tempo di calcolo: 1.83 sec


In [6]:
import pandas as pd
import numpy as np
import time

if 'df_sweeps' not in globals() or 'df' not in globals():
    raise NameError("Errore: Variabili non trovate. Esegui la cella nel notebook originale!")

print("Avvio Grid Search su Imbalance (Test Pullback vs Market Order)...")
start_opt = time.time()

# Parametri da testare
pullback_list = [0.00, 0.25, 0.50]  # 0.00 significa ENTRATA IMMEDIATA al picco
tp_list = [0.50, 0.75, 1.00, 1.25, 1.50]
sl_list = [0.50, 0.75, 1.00, 1.25, 1.50]
TIME_STOP_SEC = 20

risultati_imbalance = []

for PB in pullback_list:
    for TP in tp_list:
        for SL in sl_list:
            
            pnl_totale = 0
            win_count = 0
            loss_count = 0
            time_stop_count = 0
            missed_count = 0
            
            for _, sweep in df_sweeps.iterrows():
                t_burst = sweep['Datetime_UTC']
                direzione = sweep['Signal_Imbalance']
                
                # --- CALCOLO ENTRATA ---
                t_entrata = None
                prezzo_entrata = None
                
                if PB == 0.00:
                    # Entrata immediata senza aspettare ritracciamenti
                    t_entrata = t_burst
                    if direzione == 1: prezzo_entrata = sweep['ask']  # Paghiamo l'ask (spread)
                    else: prezzo_entrata = sweep['bid']               # Paghiamo il bid (spread)
                else:
                    # Logica Pullback
                    t_fine_pullback = t_burst + pd.Timedelta(seconds=3.0)
                    finestra_pullback = df.loc[t_burst : t_fine_pullback]
                    if finestra_pullback.empty: continue
                        
                    if direzione == 1:
                        prezzo_target = sweep['ask'] - PB
                        hit_pb = finestra_pullback[finestra_pullback['ask'] <= prezzo_target]
                        if not hit_pb.empty:
                            t_entrata = hit_pb.index[0]
                            prezzo_entrata = hit_pb['ask'].iloc[0]
                    elif direzione == -1:
                        prezzo_target = sweep['bid'] + PB
                        hit_pb = finestra_pullback[finestra_pullback['bid'] >= prezzo_target]
                        if not hit_pb.empty:
                            t_entrata = hit_pb.index[0]
                            prezzo_entrata = hit_pb['bid'].iloc[0]
                            
                if not t_entrata:
                    missed_count += 1
                    continue
                    
                # --- CALCOLO GESTIONE (TP/SL) ---
                t_scadenza = t_entrata + pd.Timedelta(seconds=TIME_STOP_SEC)
                finestra_trade = df.loc[t_entrata : t_scadenza]
                if finestra_trade.empty:
                    time_stop_count += 1
                    continue
                    
                if direzione == 1: # LONG
                    target = prezzo_entrata + TP
                    stop = prezzo_entrata - SL
                    hit_tp = (finestra_trade['bid'] >= target).idxmax() if (finestra_trade['bid'] >= target).any() else None
                    hit_sl = (finestra_trade['bid'] <= stop).idxmax() if (finestra_trade['bid'] <= stop).any() else None
                    
                    if hit_tp and hit_sl:
                        if hit_tp < hit_sl: pnl_totale += (target - prezzo_entrata); win_count += 1
                        else: pnl_totale += (stop - prezzo_entrata); loss_count += 1
                    elif hit_tp: pnl_totale += (target - prezzo_entrata); win_count += 1
                    elif hit_sl: pnl_totale += (stop - prezzo_entrata); loss_count += 1
                    else:
                        pnl_totale += (finestra_trade['bid'].iloc[-1] - prezzo_entrata)
                        time_stop_count += 1
                        
                elif direzione == -1: # SHORT
                    target = prezzo_entrata - TP
                    stop = prezzo_entrata + SL
                    hit_tp = (finestra_trade['ask'] <= target).idxmax() if (finestra_trade['ask'] <= target).any() else None
                    hit_sl = (finestra_trade['ask'] >= stop).idxmax() if (finestra_trade['ask'] >= stop).any() else None
                    
                    if hit_tp and hit_sl:
                        if hit_tp < hit_sl: pnl_totale += (prezzo_entrata - target); win_count += 1
                        else: pnl_totale += (prezzo_entrata - stop); loss_count += 1
                    elif hit_tp: pnl_totale += (prezzo_entrata - target); win_count += 1
                    elif hit_sl: pnl_totale += (prezzo_entrata - stop); loss_count += 1
                    else:
                        pnl_totale += (prezzo_entrata - finestra_trade['ask'].iloc[-1])
                        time_stop_count += 1
                        
            # Salvataggio risultato
            tot_eseguiti = win_count + loss_count + time_stop_count
            win_rate = (win_count / tot_eseguiti) * 100 if tot_eseguiti > 0 else 0
            
            risultati_imbalance.append({
                'Pullback_Req': PB,
                'Take_Profit': TP,
                'Stop_Loss': SL,
                'Net_PnL': round(pnl_totale, 2),
                'Win_Rate_%': round(win_rate, 2),
                'Eseguiti': tot_eseguiti,
                'Wins': win_count,
                'Losses': loss_count
            })

df_grid_imbalance = pd.DataFrame(risultati_imbalance).sort_values(by='Net_PnL', ascending=False).reset_index(drop=True)

print(f"\nGrid Search completata in {(time.time() - start_opt):.2f} secondi.")
print("Migliori 10 Configurazioni (Setup 2: Imbalance):")
display(df_grid_imbalance.head(10))

Avvio Grid Search su Imbalance (Test Pullback vs Market Order)...

Grid Search completata in 8.52 secondi.
Migliori 10 Configurazioni (Setup 2: Imbalance):


,Pullback_Req,Take_Profit,Stop_Loss,Net_PnL,Win_Rate_%,Eseguiti,Wins,Losses
0,0.5,0.75,0.50,-5.87,26.92,52,14,35
1,0.5,0.75,1.00,-7.13,32.69,52,17,20
2,0.5,0.50,1.00,-8.01,42.31,52,22,18
3,0.5,0.50,0.50,-8.50,32.69,52,17,34
4,0.5,1.00,0.50,-9.50,15.38,52,8,38
5,0.5,1.25,0.50,-9.50,11.54,52,6,39
6,0.5,0.75,1.25,-9.96,32.69,52,17,17
7,0.5,1.50,0.50,-10.00,9.62,52,5,40
8,0.5,0.75,0.75,-10.01,28.85,52,15,30
9,0.5,0.50,1.25,-10.34,42.31,52,22,15


In [7]:
import pandas as pd
import numpy as np
import time

if 'imbalance_signals' not in globals() or 'df' not in globals():
    raise NameError("Errore: Variabili non trovate. Esegui la cella nel notebook originale!")

print("Avvio Grid Search Imbalance su FUTURE PURO (Isolamento Alpha)...")
start_opt = time.time()

# --- 1. FILTRO CLUSTER (COOLDOWN) ---
COOLDOWN_SEC = 15
signals_df = imbalance_signals.reset_index()
valid_sweeps = []
last_sweep_time = None

for idx, row in signals_df.iterrows():
    if last_sweep_time is None or (row['Datetime_UTC'] - last_sweep_time).total_seconds() >= COOLDOWN_SEC:
        valid_sweeps.append(row)
        last_sweep_time = row['Datetime_UTC']

df_sweeps = pd.DataFrame(valid_sweeps)
print(f"Esplosioni Uniche da analizzare: {len(df_sweeps)}")

# --- 2. GRID SEARCH (SU PREZZO FUTURE REALE) ---
pullback_list = [0.00, 0.25, 0.50]
tp_list = [0.50, 0.75, 1.00, 1.25, 1.50]
sl_list = [0.50, 0.75, 1.00, 1.25, 1.50]
TIME_STOP_SEC = 20
COMMISSIONI_PT = 0.10 # Circa $5 a trade

risultati_future = []

for PB in pullback_list:
    for TP in tp_list:
        for SL in sl_list:
            
            pnl_totale = 0
            win_count = 0
            loss_count = 0
            time_stop_count = 0
            missed_count = 0
            
            for _, sweep in df_sweeps.iterrows():
                t_burst = sweep['Datetime_UTC']
                direzione = sweep['Signal_Imbalance']
                prezzo_picco = sweep['Price'] # Usiamo il prezzo puro battuto
                
                # --- CALCOLO ENTRATA ---
                t_entrata = None
                prezzo_entrata = None
                
                if PB == 0.00:
                    t_entrata = t_burst
                    prezzo_entrata = prezzo_picco
                else:
                    t_fine_pullback = t_burst + pd.Timedelta(seconds=3.0)
                    finestra_pullback = df.loc[t_burst : t_fine_pullback]
                    if finestra_pullback.empty: continue
                        
                    if direzione == 1:
                        prezzo_target = prezzo_picco - PB
                        hit_pb = finestra_pullback[finestra_pullback['Price'] <= prezzo_target]
                        if not hit_pb.empty:
                            t_entrata = hit_pb.index[0]
                            prezzo_entrata = hit_pb['Price'].iloc[0]
                    elif direzione == -1:
                        prezzo_target = prezzo_picco + PB
                        hit_pb = finestra_pullback[finestra_pullback['Price'] >= prezzo_target]
                        if not hit_pb.empty:
                            t_entrata = hit_pb.index[0]
                            prezzo_entrata = hit_pb['Price'].iloc[0]
                            
                if not t_entrata:
                    missed_count += 1
                    continue
                    
                # --- CALCOLO GESTIONE (TP/SL) ---
                t_scadenza = t_entrata + pd.Timedelta(seconds=TIME_STOP_SEC)
                finestra_trade = df.loc[t_entrata : t_scadenza]
                if finestra_trade.empty:
                    time_stop_count += 1
                    pnl_totale -= COMMISSIONI_PT
                    continue
                    
                if direzione == 1: # LONG
                    target = prezzo_entrata + TP
                    stop = prezzo_entrata - SL
                    hit_tp = (finestra_trade['Price'] >= target).idxmax() if (finestra_trade['Price'] >= target).any() else None
                    hit_sl = (finestra_trade['Price'] <= stop).idxmax() if (finestra_trade['Price'] <= stop).any() else None
                    
                    if hit_tp and hit_sl:
                        if hit_tp < hit_sl: pnl_totale += (TP - COMMISSIONI_PT); win_count += 1
                        else: pnl_totale -= (SL + COMMISSIONI_PT); loss_count += 1
                    elif hit_tp: pnl_totale += (TP - COMMISSIONI_PT); win_count += 1
                    elif hit_sl: pnl_totale -= (SL + COMMISSIONI_PT); loss_count += 1
                    else:
                        pnl_totale += (finestra_trade['Price'].iloc[-1] - prezzo_entrata) - COMMISSIONI_PT
                        time_stop_count += 1
                        
                elif direzione == -1: # SHORT
                    target = prezzo_entrata - TP
                    stop = prezzo_entrata + SL
                    hit_tp = (finestra_trade['Price'] <= target).idxmax() if (finestra_trade['Price'] <= target).any() else None
                    hit_sl = (finestra_trade['Price'] >= stop).idxmax() if (finestra_trade['Price'] >= stop).any() else None
                    
                    if hit_tp and hit_sl:
                        if hit_tp < hit_sl: pnl_totale += (TP - COMMISSIONI_PT); win_count += 1
                        else: pnl_totale -= (SL + COMMISSIONI_PT); loss_count += 1
                    elif hit_tp: pnl_totale += (TP - COMMISSIONI_PT); win_count += 1
                    elif hit_sl: pnl_totale -= (SL + COMMISSIONI_PT); loss_count += 1
                    else:
                        pnl_totale += (prezzo_entrata - finestra_trade['Price'].iloc[-1]) - COMMISSIONI_PT
                        time_stop_count += 1
                        
            # Salvataggio risultato
            tot_eseguiti = win_count + loss_count + time_stop_count
            win_rate = (win_count / tot_eseguiti) * 100 if tot_eseguiti > 0 else 0
            
            risultati_future.append({
                'Pullback_Req': PB,
                'Take_Profit': TP,
                'Stop_Loss': SL,
                'Net_PnL': round(pnl_totale, 2),
                'Win_Rate_%': round(win_rate, 2),
                'Eseguiti': tot_eseguiti,
                'Wins': win_count,
                'Losses': loss_count
            })

df_grid_future = pd.DataFrame(risultati_future).sort_values(by='Net_PnL', ascending=False).reset_index(drop=True)

print(f"\nGrid Search su Future completata in {(time.time() - start_opt):.2f} secondi.")
print("Migliori 10 Configurazioni (Isolamento Alpha su FUTURE PURO):")
display(df_grid_future.head(10))

Avvio Grid Search Imbalance su FUTURE PURO (Isolamento Alpha)...
Esplosioni Uniche da analizzare: 263

Grid Search su Future completata in 12.56 secondi.
Migliori 10 Configurazioni (Isolamento Alpha su FUTURE PURO):


,Pullback_Req,Take_Profit,Stop_Loss,Net_PnL,Win_Rate_%,Eseguiti,Wins,Losses
0,0.50,1.00,1.50,79.10,76.09,184,140,26
1,0.50,0.75,1.50,78.10,88.04,184,162,16
2,0.50,0.75,1.25,76.10,86.41,184,159,19
3,0.50,0.75,1.00,75.10,84.78,184,156,24
4,0.50,1.00,1.00,74.10,71.74,184,132,40
5,0.50,1.00,1.25,72.60,73.37,184,135,33
6,0.50,1.25,1.50,66.35,63.04,184,116,39
7,0.50,1.25,1.00,66.10,58.70,184,108,54
8,0.25,0.50,1.50,63.10,92.37,249,230,18
9,0.50,1.25,1.25,61.85,60.33,184,111,47
